In [4]:
# Verify the versions
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.__version__, np.__version__

('2.3.3', '2.4.6')

In [5]:
# Enable logging.
import logging
from pathlib import Path

logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("pipeline.log", encoding="utf-8")
    ],
    force=True
)

logger.info("Logging has been enabled.")

2026-10-10 00:19:33,642 | INFO | __main__ | Logging has been enabled.


In [6]:
# 1.	Load the raw CSV file 
def load_raw_data(file_path):
    data = pd.read_csv(file_path, keep_default_na=False)

    logger.info(
        "Raw CSV has been loaded succesfully: %s rows and %s columns.",
        data.shape[0],
        data.shape[1]
    )

    return data


raw_path = Path("../part1_data_analytics/Metro_Interstate_Traffic_Volume.csv")

try:
    raw_data = load_raw_data(raw_path)

except (OSError, UnicodeError, pd.errors.ParserError,
        pd.errors.EmptyDataError):
    raw_data = None
    logger.error("Could not load the raw CSV", exc_info=True)

2026-10-10 00:19:48,334 | INFO | __main__ | Raw CSV has been loaded succesfully: 48204 rows and 9 columns.


In [7]:
# Columns headers from CSV file
EXPECTED_COLUMNS = [
    "holiday",
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "weather_main",
    "weather_description",
    "date_time",
    "traffic_volume"
]


# Check the columns before cleaning the data.

def validate_schema(data):
    missing_columns = [
        column for column in EXPECTED_COLUMNS
        if column not in data.columns
    ]

    # Stop validation if any required column is missing.
    if missing_columns:
        raise ValueError(
            f"Required columns are missing: {missing_columns}."
        )

    logger.info("Schema check has passed all validations: all required columns are present.")


# Cleaning can start only after the schema check passes.
schema_valid = False

if raw_data is not None:
    try:
        validate_schema(raw_data)
        schema_valid = True

    except ValueError:
        logger.error("Schema check failed", exc_info=True)
else:
    logger.error("Schema check cannot be run because data wa not loaded.")

2026-10-10 00:27:26,453 | INFO | __main__ | Schema check has passed all validations: all required columns are present.


In [8]:
# Check  text categories before changing any values

category_columns = [
    "holiday",
    "weather_main",
    "weather_description"
]

if schema_valid:
    for column in category_columns:
        # Show each different value oce.
        display({
            "column": column,
            "values": raw_data[column].unique().tolist()
        })
else:
    logger.error("Cannot inspect categories because the schema chec has been k failed")

{'column': 'holiday',
 'values': ['None',
  'Columbus Day',
  'Veterans Day',
  'Thanksgiving Day',
  'Christmas Day',
  'New Years Day',
  'Washingtons Birthday',
  'Memorial Day',
  'Independence Day',
  'State Fair',
  'Labor Day',
  'Martin Luther King Jr Day']}

{'column': 'weather_main',
 'values': ['Clouds',
  'Clear',
  'Rain',
  'Drizzle',
  'Mist',
  'Haze',
  'Fog',
  'Thunderstorm',
  'Snow',
  'Squall',
  'Smoke']}

{'column': 'weather_description',
 'values': ['scattered clouds',
  'broken clouds',
  'overcast clouds',
  'sky is clear',
  'few clouds',
  'light rain',
  'light intensity drizzle',
  'mist',
  'haze',
  'fog',
  'proximity shower rain',
  'drizzle',
  'moderate rain',
  'heavy intensity rain',
  'proximity thunderstorm',
  'thunderstorm with light rain',
  'proximity thunderstorm with rain',
  'heavy snow',
  'heavy intensity drizzle',
  'snow',
  'thunderstorm with heavy rain',
  'freezing rain',
  'shower snow',
  'light rain and snow',
  'light intensity shower rain',
  'SQUALLS',
  'thunderstorm with rain',
  'proximity thunderstorm with drizzle',
  'thunderstorm',
  'Sky is Clear',
  'very heavy rain',
  'thunderstorm with light drizzle',
  'light snow',
  'thunderstorm with drizzle',
  'smoke',
  'shower drizzle',
  'light shower snow',
  'sleet']}

In [10]:
# Cleanr the data.

def standardise_categories(data):
    cleaned = data.copy()

    for column in category_columns:
        original = cleaned[column]

        # Remove spaces at the start and end of each value.
        standardised = original.str.strip()

        # Use lowercase for detailed weather descriptions.
        if column == "weather_description":
            standardised = standardised.str.lower()
        else:
            standardised = standardised.str.title()

        # Count the rows whose text values have been changed.
        changed_rows = int(
            original.ne(standardised).fillna(False).sum()
        )

        cleaned[column] = standardised

        if changed_rows > 0:
            logger.warning(
                "%s: modified %s rows to standardise spaces and capitalisation.",
                column,
                changed_rows
            )
        else:
            logger.info("%s: no category changes needed.", column)

    return cleaned


if schema_valid:
    cleaned_data = standardise_categories(raw_data)
else:
    logger.error("Cannot clean the categories because the schema check has been failed.")

2026-10-10 00:31:46,601 | INFO | __main__ | holiday: no category changes needed.
2026-10-10 00:31:46,623 | INFO | __main__ | weather_main: no category changes needed.
2026-10-10 00:31:46,646 | WARNING | __main__ | weather_description: modified 1730 rows to standardise spaces and capitalisation.


In [11]:
# Convert date/time text and remove rows with invalid dates.

def clean_datetime(data):
    cleaned = data.copy()

    parsed_dates = pd.to_datetime(
        cleaned["date_time"],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce"
    )

   
    invalid_dates = parsed_dates.isna()
    invalid_count = int(invalid_dates.sum())

    cleaned["date_time"] = parsed_dates

    if invalid_count > 0:
        cleaned = cleaned.loc[~invalid_dates].copy()

        logger.warning(
            "Dropped %s rows because date_time was missing or invalid.",
            invalid_count
        )
    else:
        logger.info("Date/time check has passed: no missing or invalid dates.")

    logger.info("date_time is converted to datetime format.")

    return cleaned


if schema_valid:
    cleaned_data = clean_datetime(cleaned_data)
else:
    logger.error("Cannot clean dates because the schema check has been failed.")

2026-10-10 00:34:15,887 | INFO | __main__ | Date/time check has passed: no missing or invalid dates.
2026-10-10 00:34:15,888 | INFO | __main__ | date_time is converted to datetime format.


In [12]:
# Identify and Remove the rows that repeat exactly across every column.

def remove_duplicates(data):
    duplicate_count = int(data.duplicated().sum())

    cleaned = data.drop_duplicates().copy()

    if duplicate_count > 0:
        logger.warning(
            "Dropped %s rows because all column values were duplicates.",
            duplicate_count
        )
    else:
        logger.info("Duplicate check completed: no exact duplicates found.")

    logger.info(
        "Dataset after duplicate removal: %s rows and %s columns.",
        cleaned.shape[0],
        cleaned.shape[1]
    )

    return cleaned


if schema_valid:
    cleaned_data = remove_duplicates(cleaned_data)
else:
    logger.error("Cannot remove duplicates because the schema check failed.")

2026-10-10 00:36:51,765 | WARNING | __main__ | Dropped 17 rows because all column values were duplicates.
2026-10-10 00:36:51,766 | INFO | __main__ | Dataset after duplicate removal: 48187 rows and 9 columns.


In [13]:
# Inspect numerical columns for missing and unusual values.
numeric_columns = [
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "traffic_volume"
]

if schema_valid:
    display(cleaned_data[numeric_columns].describe())

    # Count the two example problems mentioned in the assignment.
    display({
        "temperature_at_or_below_0_kelvin":
            int((cleaned_data["temp"] <= 0).sum()),
        "rainfall_above_9000_mm":
            int((cleaned_data["rain_1h"] > 9000).sum())
    })
else:
    logger.error("Cannot inspect numerical values because the schema check failed.")

,temp,rain_1h,snow_1h,clouds_all,traffic_volume
count,48187.000000,48187.000000,48187.000000,48187.000000,48187.000000
mean,281.204995,0.334382,0.000222,49.365451,3259.618134
std,13.338738,44.797033,0.008169,39.015213,1986.954465
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,272.160000,0.000000,0.000000,1.000000,1192.500000
50%,282.450000,0.000000,0.000000,64.000000,3379.000000
75%,291.806000,0.000000,0.000000,90.000000,4933.000000
max,310.070000,9831.300000,0.510000,100.000000,7280.000000


{'temperature_at_or_below_0_kelvin': 10, 'rainfall_above_9000_mm': 1}

In [14]:
#Replace invalid temperatures using valid readings from the same month.
def fix_temperature(data):
    cleaned = data.copy()

    # Find missing, non-finite or impossible Kelvin readings.
    invalid = (
        cleaned["temp"].isna()
        | ~np.isfinite(cleaned["temp"])
        | (cleaned["temp"] <= 0)
    )

    months = cleaned["date_time"].dt.month

    # Check each calendar month separately.
    for month in sorted(months.unique()):
        month_rows = months == month
        rows_to_fix = month_rows & invalid
        count = int(rows_to_fix.sum())

        if count == 0:
            continue


        monthly_median = cleaned.loc[
            month_rows & ~invalid, "temp"
        ].median()

        if pd.isna(monthly_median):
            raise ValueError(
                f"No valid temperature readings for month {month}."
            )

        logger.debug(
            "Month %s temperature median is %s Kelvin.",
            month,
            monthly_median
        )

        cleaned.loc[rows_to_fix, "temp"] = monthly_median

        logger.warning(
            "Fixed %s missing or invalid temperature readings in month %s "
            "using the monthly median.",
            count,
            month
        )

    if not invalid.any():
        logger.info("Temperature check passed. No changes needed.")

    return cleaned


# Record an error if the temperature step cannot finish.
if schema_valid:
    try:
        cleaned_data = fix_temperature(cleaned_data)
    except ValueError:
        logger.error("Could not fix temperature readings.", exc_info=True)
else:
    logger.error("Cannot check temperatures because the schema check failed.")

2026-10-10 00:39:59,824 | WARNING | __main__ | Fixed 4 missing or invalid temperature readings in month 1 using the monthly median.
2026-10-10 00:39:59,833 | WARNING | __main__ | Fixed 6 missing or invalid temperature readings in month 2 using the monthly median.


In [15]:
# Replace invaid rainfll using valid readings from the same month.

def fix_rainfall(data):
    cleaned = data.copy()

    # Rainfall must be finite and between 0 and 9000 mm.
    invalid = (
        cleaned["rain_1h"].isna()
        | ~np.isfinite(cleaned["rain_1h"])
        | (cleaned["rain_1h"] < 0)
        | (cleaned["rain_1h"] > 9000)
    )

    months = cleaned["date_time"].dt.month

    #Find a separat median for each calendar month.
    for month in sorted(months.unique()):
        month_rows = months == month
        rows_to_fix = month_rows & invalid
        count = int(rows_to_fix.sum())

        if count == 0:
            continue

        # Do not include invalid rainfall in the median.
        monthly_median = cleaned.loc[
            month_rows & ~invalid, "rain_1h"
        ].median()

        if pd.isna(monthly_median):
            raise ValueError(
                f"No valid rainfall readings for month {month}."
            )

        logger.debug(
            "Month %s rainfall median is %s mm.",
            month,
            monthly_median
        )

        cleaned.loc[rows_to_fix, "rain_1h"] = monthly_median

        logger.warning(
            "Fixed %s missing or invalid rainfall readings in month %s "
            "using the monthly median.",
            count,
            month
        )

    if not invalid.any():
        logger.info("Rainfall check passed. No changes needed.")

    return cleaned


# Record an error the rainfall step cannot finish.
if schema_valid:
    try:
        cleaned_data = fix_rainfall(cleaned_data)
    except ValueError:
        logger.error("Could not fix rainfall readings.", exc_info=True)
else:
    logger.error("Cannot check rainfall because the schema check failed.")

2026-10-10 00:41:26,760 | WARNING | __main__ | Fixed 1 missing or invalid rainfall readings in month 7 using the monthly median.


In [16]:
# Count missing cells and text containing only spaces.
def check_missing_values(data):
    missing_counts = {}

    for column in data.columns:
        missing = data[column].isna()

        # Blank text also counts as missing.
        if pd.api.types.is_object_dtype(data[column]) or \
                pd.api.types.is_string_dtype(data[column]):
            blank = data[column].astype("string").str.strip().eq("")
            missing = missing | blank.fillna(False)

        missing_counts[column] = int(missing.sum())

    logger.info("Missing value check completed.")

    return pd.Series(missing_counts, name="missing_rows")


if schema_valid:
    display(check_missing_values(cleaned_data))
else:
    logger.error("Cannot check missing values because the schema check failed.")

2026-10-10 00:42:19,846 | INFO | __main__ | Missing value check completed.


holiday                0
temp                   0
rain_1h                0
snow_1h                0
clouds_all             0
weather_main           0
weather_description    0
date_time              0
traffic_volume         0
Name: missing_rows, dtype: int64

In [17]:
# Check the remaining numerical fields against simple valid ranges.
if schema_valid:
    invalid_counts = {
        "invalid_snow": int((
            ~np.isfinite(cleaned_data["snow_1h"])
            | (cleaned_data["snow_1h"] < 0)
        ).sum()),

        "invalid_cloud_cover": int((
            ~np.isfinite(cleaned_data["clouds_all"])
            | (cleaned_data["clouds_all"] < 0)
            | (cleaned_data["clouds_all"] > 100)
        ).sum()),

        "invalid_traffic_volume": int((
            ~np.isfinite(cleaned_data["traffic_volume"])
            | (cleaned_data["traffic_volume"] < 0)
            | (cleaned_data["traffic_volume"] % 1 != 0)
        ).sum())
    }

    logger.info("Snow, cloud cover and traffic checks completed.")
    display(invalid_counts)
else:
    logger.error("Cannot check numerical fields because the schema check failed.")

2026-10-10 00:42:53,155 | INFO | __main__ | Snow, cloud cover and traffic checks completed.


{'invalid_snow': 0, 'invalid_cloud_cover': 0, 'invalid_traffic_volume': 0}

In [18]:
# Check that the cleaned data has no remaining problems.



if schema_valid:
    checks = {
        "no_missing_values":
            check_missing_values(cleaned_data).eq(0).all(),

        "no_exact_duplicates":
            not cleaned_data.duplicated().any(),

        "valid_dates":
            cleaned_data["date_time"].notna().all(),

        "valid_temperatures": (
            np.isfinite(cleaned_data["temp"])
            & (cleaned_data["temp"] > 0)
        ).all(),

        "valid_rainfall": (
            np.isfinite(cleaned_data["rain_1h"])
            & cleaned_data["rain_1h"].between(0, 9000)
        ).all(),

        "other_numeric_checks_passed":
            all(count == 0 for count in invalid_counts.values())
    }

    
    cleaning_valid = all(checks.values())

    if cleaning_valid:
        logger.info(
            "Cleaned data checks passed. There are %s rows and %s columns.",
            cleaned_data.shape[0],
            cleaned_data.shape[1]
        )
    else:
        logger.error("Cleaned data checks failed. Do not save the data yet.")

    display(pd.Series(checks, name="passed"))
else:
    cleaning_valid = False
    logger.error("Cannot check cleaned data because the schema check failed.")

2026-10-10 00:43:54,675 | INFO | __main__ | Missing value check completed.
2026-10-10 00:43:54,705 | INFO | __main__ | Cleaned data checks passed. There are 48187 rows and 9 columns.


no_missing_values              True
no_exact_duplicates            True
valid_dates                    True
valid_temperatures             True
valid_rainfall                 True
other_numeric_checks_passed    True
Name: passed, dtype: bool

In [19]:
#Save the cleaned data only after all ches pass.

if cleaning_valid:
    try:
        output_path = Path("data/traffic_cleaned.csv")
        output_path.parent.mkdir(parents=True, exist_ok=True)

        # Do not save the DataFrame index as an extra column.
        cleaned_data.to_csv(output_path, index=False)

        logger.info("Cleaned data saved to %s.", output_path.as_posix())

    except OSError:
        logger.error("Could not save the cleaned data.", exc_info=True)
else:
    logger.error("Data was not saved because the cleaning checks failed.")

2026-10-10 00:44:56,628 | INFO | __main__ | Cleaned data saved to data/traffic_cleaned.csv.
